# 08.03 · Interrupts and human-in-the-loop (graph level)

**Module 08 · LangGraph Production Features** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 120–150 min |
| 📋 Prerequisites | Lessons 08.01–08.02; Lesson 04.04 (`HumanInTheLoopMiddleware` for agents). |
| 🔑 Keys | none (optional: `GROQ_API_KEY` for the live draft in Part 6) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Pause** a graph anywhere with `interrupt(payload)` and **resume** it with `Command(resume=value)`.
2. **Build** approve / edit / reject flows, including a validation loop for bad answers.
3. **Follow** the interrupt rules: the node re-runs from its top, no `try/except` around `interrupt`, a stable order, idempotent code before it.
4. **Handle** interrupts inside tools and several interrupts at once (resume by id), and drive it all from notebook cells with live-updating output.

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts) | `interrupt()`, `Command(resume=...)`, approve/edit/validate patterns, the rules, multiple interrupts |
| [Persistence](https://docs.langchain.com/oss/python/langgraph/persistence) | Why interrupts need a checkpointer and a `thread_id` |
| [Human-in-the-loop (agents)](https://docs.langchain.com/oss/python/langchain/human-in-the-loop) | The middleware from Module 04 is built on these interrupts |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2" "langchain-groq>=1.1.3,<2"

**You should now see** nothing, or *"Note: you may need to restart the kernel to use updated packages."* Restart only if a package was **upgraded** after you had already imported it (Lesson 00.07). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.03).

In [ ]:
from importlib import metadata

for package in ("langgraph", "langchain-core", "langchain-groq",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/08-langgraph-production-features/).
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import the course package (installed editable in Lesson 00.02; src/ fallback otherwise).
try:
    import mastery
except ModuleNotFoundError:
    sys.path.insert(0, str(REPO_ROOT / "src"))
    import mastery
    print("ℹ️ Loaded mastery from src/. Fix once: python -m pip install -e \".[dev]\"")
print("Repo root:", REPO_ROOT)

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Step 4: Silence known, harmless noise so real problems stand out.
import os
import warnings

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (Project 08 can reuse Module 06): no usage telemetry.
# Same as Lesson 02.04: langchain-core 1.6.9/1.6.10 Pydantic serializer warning about the injected runtime.
warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

---
## Part 1 · `interrupt()` and `Command(resume=...)`

`interrupt(value)` stops the run **inside a node**, saves a checkpoint, and hands `value` (anything JSON-serializable) to the caller. The caller later sends an answer with `Command(resume=answer)`; the node **starts again from its first line**, and this time `interrupt(...)` returns the answer.

In [ ]:
# 🧪 Offline: one question to a human.
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt
from typing_extensions import TypedDict


class Order(TypedDict, total=False):
    item: str
    quantity: int
    status: str


runs = {"ask": 0}


def ask_quantity(state: Order) -> dict:
    runs["ask"] += 1                                         # counts how often this node STARTS
    answer = interrupt({"question": f"How many {state['item']}?", "hint": "a whole number"})
    return {"quantity": int(answer)}


def place(state: Order) -> dict:
    return {"status": f"ordered {state['quantity']} x {state['item']}"}


b = StateGraph(Order)
b.add_node("ask_quantity", ask_quantity)
b.add_node("place", place)
b.add_edge(START, "ask_quantity")
b.add_edge("ask_quantity", "place")
b.add_edge("place", END)
shop = b.compile(checkpointer=InMemorySaver())                 # interrupts NEED a checkpointer

cfg = {"configurable": {"thread_id": "order-1"}}
paused = shop.invoke({"item": "sensor mounts"}, cfg)
print("paused with:", paused["__interrupt__"])
print("next       :", shop.get_state(cfg).next)

**You should now see** `paused with: [Interrupt(value={'question': 'How many sensor mounts?', 'hint': 'a whole number'}, id='...', response_schema=None)]` and `next       : ('ask_quantity',)`.

The result has an extra key, `__interrupt__`: a list of `Interrupt` objects (the payload plus an **id**). The thread is saved and waiting; your program can stop, and a different process can resume it hours later (that's why the checkpointer is required).

In [ ]:
# 🧪 Offline: resume with the human's answer.
done = shop.invoke(Command(resume="3"), cfg)
print(done["status"])
print("ask_quantity started", runs["ask"], "times")

**You should now see** `ordered 3 x sensor mounts` and `ask_quantity started 2 times`.

### What just happened?

The node ran **twice**: once up to the interrupt, once from the top with `interrupt(...)` returning `"3"`. Everything **before** the `interrupt` line ran again. That is the single most important fact about interrupts, and it gives us the rules.

| Rule (from the docs) | Why |
|---|---|
| Code before `interrupt()` must be safe to repeat (no emails, no charges, no DB inserts) | It re-runs on resume |
| Don't wrap `interrupt()` in `try/except Exception` | It works by raising a special exception; catching it breaks the pause |
| Several `interrupt()` calls in one node: same order every time, no conditional skipping | Answers are matched to calls **by position** |
| Payloads and answers must be JSON-serializable | They are checkpointed and sent over HTTP by the server |

---
## Part 2 · Approve, edit, reject

The classic review step. `mastery.graphs.hitl.approval_node(...)` builds it once for the whole course:

- pauses with `{"kind": "approval", "draft": ..., "actions": ["approve", "edit", "reject"], ...}`;
- **approve** → go to `approved`; **edit** `{"action": "edit", "text": ...}` → replace the draft, go to `approved`; **reject** `{"action": "reject", "feedback": ...}` → go back to `rejected` with the feedback in state;
- after `max_rejections` rejections it gives up (`give_up` node) instead of looping forever;
- an **invalid answer** re-asks in the same node with an `error` field (a validation loop), so a typo never crashes the run.

It returns a `Command(goto=...)`, so register it with `destinations=` for a correct drawing.

In [ ]:
# 🧪 Offline: write → approval → publish, with a rewrite loop.
from mastery.graphs.hitl import approval_node, resume, run_until_pause


class Post(TypedDict, total=False):
    topic: str
    draft: str
    feedback: str
    rejections: int
    approval: str
    final: str


def write(state: Post) -> dict:
    version = state.get("rejections", 0) + 1
    note = f" (revised: {state['feedback']})" if state.get("feedback") else ""
    return {"draft": f"v{version}: Five tips about {state['topic']}{note}"}


def publish(state: Post) -> dict:
    return {"final": state["draft"] + " [published]"}


b = StateGraph(Post)
b.add_node("write", write)
b.add_node("approval", approval_node(approved="publish", rejected="write", max_rejections=3),
           destinations=("publish", "write", END))
b.add_node("publish", publish)
b.add_edge(START, "write")
b.add_edge("write", "approval")
b.add_edge("publish", END)
review = b.compile(checkpointer=InMemorySaver())

rcfg = {"configurable": {"thread_id": "post-1"}}
first = run_until_pause(review, {"topic": "winter battery care"}, rcfg)
print("draft to review:", first["interrupts"][0].value["draft"])

**You should now see** `✅ write    updated: draft`, then `⏸️  paused → {'kind': 'approval', ...}` and `draft to review: v1: Five tips about winter battery care`.

`run_until_pause` streams node updates (one line each) and returns `{"values", "interrupts", "done"}`.

### Answer from a parameter cell

Change `DECISION` and re-run **the two cells below** to try each path. The first run uses the scripted sequence (reject, then a typo, then approve) so Restart & Run All always finishes.

In [ ]:
# Parameters: what the "human" answers, in order. Try {"action": "edit", "text": "My own text"}.
DECISIONS = [
    {"action": "reject", "feedback": "add a safety warning"},
    {"action": "reject"},                     # invalid: no feedback -> the node asks again
    "approve",
]

In [ ]:
# 🧪 Offline: answer each pause; a live-updating status line shows where we are.
from IPython.display import Markdown, display

status = display(Markdown("⏳ starting…"), display_id=True)    # one output area we keep updating
for i, decision in enumerate(DECISIONS, 1):
    result = resume(review, rcfg, decision, printer=None)
    if result["interrupts"]:
        payload = result["interrupts"][0].value
        status.update(Markdown(f"⏸️ **pause {i}**: `{payload['draft']}`" + (f"  \n❗ {payload['error']}" if payload.get("error") else "")))
        print(f"answer {i}: {decision!r:<55} -> paused again" + (f" (error: {payload['error']})" if payload.get("error") else ""))
    else:
        status.update(Markdown(f"✅ **done**: {result['values']['final']}"))
        print(f"answer {i}: {decision!r:<55} -> done")
print("approval:", result["values"]["approval"], "| rejections:", result["values"]["rejections"])

**You should now see** one status line that changed in place (it ends as `✅ done: v2: Five tips about winter battery care (revised: add a safety warning) [published]`) and three printed lines:

- answer 1 (reject with feedback) → paused again (the v2 draft);
- answer 2 (`{'action': 'reject'}`) → paused again `(error: ... reject needs non-empty 'feedback')`;
- answer 3 (`'approve'`) → done; then `approval: approved | rejections: 1`.

`display(..., display_id=True)` returns a handle; `.update(...)` rewrites the same output instead of printing a new one. Use it for status lines, progress and "waiting for you" banners.

In [ ]:
# 🧪 Offline: the give-up path. Three rejections → stop looping, no publish.
from mastery.graphs.hitl import drive, scripted_reviewer

stubborn = scripted_reviewer([{"action": "reject", "feedback": "no"}] * 5)
out = drive(review, {"topic": "mounting"}, {"configurable": {"thread_id": "post-2"}}, stubborn)
print("approval:", out["values"]["approval"], "| rejections:", out["values"]["rejections"],
      "| published:", "final" in out["values"], "| pauses:", out["pauses"])

**You should now see** `approval: gave_up | rejections: 3 | published: False | pauses: 3`. `drive(graph, inputs, cfg, reviewer)` answers every pause with `reviewer(payload)`: the scripted reviewer is a fake human for tests and keyless cells; in an app it's your UI.

---
## Part 3 · Interrupts inside a tool

A tool can ask for confirmation itself: call `interrupt()` in the tool function. The pause bubbles up through `ToolNode` to the graph. Here a scripted model asks to send an email; the tool waits for "yes".

In [ ]:
# 🧪 Offline: model → tools; the email tool needs human confirmation.
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from mastery.testing import ScriptedChatModel

SENT = []


@tool
def send_email(to: str, subject: str) -> str:
    """Send an email (asks a human first)."""
    answer = interrupt({"kind": "confirm", "action": f"send '{subject}' to {to}", "reply": "yes / no"})
    if str(answer).strip().lower() != "yes":
        return "Cancelled by the user. Do not retry."
    SENT.append((to, subject))                                  # the side effect is AFTER the interrupt
    return f"Sent '{subject}' to {to}."


model = ScriptedChatModel(responses=[
    AIMessage("", tool_calls=[{"name": "send_email", "args": {"to": "ana@example.com", "subject": "Battery recall"}, "id": "call_1"}]),
    AIMessage("Done: the email was sent."),
])


def agent(state: MessagesState) -> dict:
    return {"messages": [model.bind_tools([send_email]).invoke(state["messages"])]}


b = StateGraph(MessagesState)
b.add_node("agent", agent)
b.add_node("tools", ToolNode([send_email]))
b.add_edge(START, "agent")
b.add_conditional_edges("agent", tools_condition)
b.add_edge("tools", "agent")
mailer = b.compile(checkpointer=InMemorySaver())

mcfg = {"configurable": {"thread_id": "mail-1"}}
paused = mailer.invoke({"messages": [HumanMessage("Tell Ana about the recall.")]}, mcfg)
print("asks:", paused["__interrupt__"][0].value["action"], "| sent so far:", SENT)
final = mailer.invoke(Command(resume="yes"), mcfg)
print("tool said:", final["messages"][-2].content)
print("model said:", final["messages"][-1].content, "| sent:", SENT)

**You should now see** `asks: send 'Battery recall' to ana@example.com | sent so far: []`, then `tool said: Sent 'Battery recall' to ana@example.com.` and `model said: Done: the email was sent. | sent: [('ana@example.com', 'Battery recall')]`.

The send happens **after** the interrupt, so it runs once. Module 04's `HumanInTheLoopMiddleware` does the same for `create_agent` without changing your tools; a graph-level interrupt is for when you own the graph.

---
## Part 4 · Several interrupts at once

Two branches running in the same step can both pause. The result then holds **two** interrupts, and you resume with a dict `{interrupt_id: answer}`.

In [ ]:
# 🧪 Offline: two reviewers in parallel (legal and tech), answered together.
import operator
from typing import Annotated


class Review(TypedDict, total=False):
    notes: Annotated[list[str], operator.add]


def reviewer(name):
    def node(state: Review) -> dict:
        verdict = interrupt({"kind": "review", "reviewer": name})
        return {"notes": [f"{name}: {verdict}"]}
    return node


b = StateGraph(Review)
for name in ("legal", "tech"):
    b.add_node(name, reviewer(name))
    b.add_edge(START, name)
both = b.compile(checkpointer=InMemorySaver())
pcfg = {"configurable": {"thread_id": "parallel-1"}}
pending = both.invoke({}, pcfg)["__interrupt__"]
print("pending:", sorted(i.value["reviewer"] for i in pending))
answers = {i.id: ("ok" if i.value["reviewer"] == "legal" else "needs a diagram") for i in pending}
print(sorted(both.invoke(Command(resume=answers), pcfg)["notes"]))

**You should now see** `pending: ['legal', 'tech']` and `['legal: ok', 'tech: needs a diagram']`. Match answers by `id`, never by list position: the order of parallel interrupts isn't guaranteed.

---
## Part 5 · Static breakpoints (debugging only)

`compile(interrupt_before=["node"])` (or `interrupt_after=`) pauses **before** a node without changing its code. Resume with `invoke(None, cfg)`. The docs recommend them for **debugging** (stepping through a graph, like Studio does), not for human review: use `interrupt()` for that, because it carries a payload and an answer.

In [ ]:
# 🧪 Offline: pause before `place`, inspect, continue.
b_shop = StateGraph(Order)
b_shop.add_node("place", place)
b_shop.add_edge(START, "place")
stepper = b_shop.compile(checkpointer=InMemorySaver(), interrupt_before=["place"])
scfg = {"configurable": {"thread_id": "debug-1"}}
stepper.invoke({"item": "cables", "quantity": 2}, scfg)
print("stopped before:", stepper.get_state(scfg).next)
print(stepper.invoke(None, scfg)["status"])

**You should now see** `stopped before: ('place',)` and `ordered 2 x cables`.

---
## Part 6 · The Project 08 graph, paused twice

Project 08's research agent (`mastery.graphs.research`) pauses at `clarify_question` when a request is vague and at `approval` before finalizing. With offline models:

In [ ]:
# 🧪 Offline: a vague question → clarify → approve. The graph has async nodes, so we await.
from mastery.graphs.hitl import adrive
from mastery.graphs.research import build_research_graph, corpus_docs_search, offline_models, offline_web_search
from mastery.graphs.state import memory_checkpointer

agent = build_research_graph(offline_models(), web_search=offline_web_search(), docs_search=corpus_docs_search(),
                             checkpointer=memory_checkpointer())
seen = []
def human(payload):
    seen.append(payload["kind"])
    return "battery care in winter" if payload["kind"] == "clarify" else "approve"

out = await adrive(agent, {"question": "Battery?"}, {"configurable": {"thread_id": "agent-1"}}, human)
print("pauses:", seen)
print(out["values"]["final"][:400])

**You should now see** `pauses: ['clarify', 'approval']` and a short answer about winter battery care with `[source: support/faq-battery.md]` markers, ending with a `Sources:` line.

> 💸 Calls `gpt-oss-20b` 3 times (clarify, plan, draft), a few thousand tokens of your free-tier quota. Without `GROQ_API_KEY` it prints `⏭️ Skipped`.

In [ ]:
# 💸 Live: the same graph with gpt-oss-20b; the approval comes from the scripted "human".
if HAVE_GROQ_KEY:
    from mastery.models import get_model

    llm = get_model("fast", temperature=0, reasoning_effort="low")
    live = build_research_graph({"clarifier": llm, "planner": llm, "writer": llm}, web_search=offline_web_search(),
                                docs_search=corpus_docs_search(), checkpointer=memory_checkpointer())
    live_out = await adrive(live, {"question": "How do I keep the battery healthy in winter?"},
                            {"configurable": {"thread_id": "live-1"}}, human)
    print("pauses:", live_out["pauses"])
    print(live_out["values"]["final"][:600])
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** (with a key) `pauses: 1` (the question is clear, so only the approval; `2` if the model still asks a clarifying question, which `human` answers) and a model-written answer citing the battery FAQ. Without a key: `⏭️ Skipped`.

> 💡 In `langgraph dev` / Studio, the same pauses show up as an **interrupt panel**: you type the JSON answer (`{"action": "approve"}`) there. `apps/m08_research_agent/` is this graph as an app (Project 08).

---
## ⚠️ Pitfalls and troubleshooting (Windows Git Bash included)

| Symptom | Cause | Fix |
|---|---|---|
| `ValueError: No checkpointer set` / interrupt does nothing | Compiled without a checkpointer | `compile(checkpointer=...)` (the server provides one) |
| An email/DB write happened twice | Side effect **before** `interrupt()` | Move it after, or into the next node |
| The pause "disappears" | `try: ... except Exception:` around `interrupt()` | Catch only your own errors; let `GraphInterrupt` through |
| Answers end up in the wrong place | Conditional or reordered `interrupt()` calls in one node | Same calls, same order; or split into nodes |
| `Command(resume=...)` starts a new run | Different `thread_id` | Resume on the **same** config |
| Resuming with a list for parallel interrupts | Answers must be keyed | `{interrupt.id: answer}` |
| `TypeError: Object of type ... is not JSON serializable` | Non-JSON payload | Send dicts/strings; `model_dump()` Pydantic objects |

## ⚠️ Production notes

> ⚠️ **Production note: a pause can last days.**
> Store threads in a durable checkpointer (Postgres, or the server's), show the payload in your UI, and set a policy for abandoned approvals (remind, expire, or auto-reject after N days).

> ⚠️ **Production note: validate the human too.**
> Treat resume values like any user input: validate them (the validation loop), log who approved what (the answer is in the checkpoint history), and never let a free-text "edit" bypass your output guardrails (Module 04).

## 🏋️ Exercises

1. **(Easy)** In Part 1, resume with `Command(resume="three")`. What happens, and how would you fix it?
2. **(Medium)** Build a node that asks **two** questions in a row (name, then email) with two `interrupt()` calls. How many times does the node start in total?
3. **(Medium)** Add an `"escalate"` action to the review flow by writing your own approval node (copy the idea of `approval_node`) that routes to an `escalate` node.
4. **(Stretch)** Make `send_email` (Part 3) accept an **edited** subject: the human answers `{"approve": true, "subject": "..."}`.

<details>
<summary><b>Show solutions</b></summary>

**1.** `int("three")` raises `ValueError` **after** the resume, so the run fails (the thread stays at `ask_quantity`; you can resume again). Fix with a validation loop:
```python
def ask_quantity(state):
    prompt = {"question": f"How many {state['item']}?"}
    while True:
        answer = interrupt(prompt)
        if str(answer).isdigit():
            return {"quantity": int(answer)}
        prompt = {**prompt, "error": f"{answer!r} is not a whole number"}
```

**2.** Three: the first start pauses at question 1; resuming starts it again, question 1 returns its answer and question 2 pauses; the third start returns both answers. Answers are matched by **position**.

**3.**
```python
def review_node(state):
    answer = interrupt({"draft": state["draft"], "actions": ["approve", "reject", "escalate"]})
    if answer == "escalate":
        return Command(goto="escalate")
    ...
builder.add_node("approval", review_node, destinations=("publish", "write", "escalate"))
```

**4.**
```python
answer = interrupt({"action": f"send '{subject}' to {to}"})
if not answer.get("approve"):
    return "Cancelled by the user."
subject = answer.get("subject", subject)
```
</details>

## ✅ Checkpoint

- [ ] I paused a graph with `interrupt()` and resumed it with `Command(resume=...)`.
- [ ] I can explain why code before `interrupt()` runs twice, and the four rules.
- [ ] I built approve / edit / reject with a validation loop and a give-up limit.
- [ ] I resumed parallel interrupts by id and paused inside a tool.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Read [Interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts) and note in [`docs/research-log.md`](../../docs/research-log.md): how the docs recommend resuming **multiple** interrupts, what they say about using interrupts inside **subgraphs** (Lesson 08.05), and how `HumanInTheLoopMiddleware` (Module 04) maps onto `interrupt()`.

## Summary

- `interrupt(payload)` pauses inside a node; `Command(resume=answer)` continues; a checkpointer and `thread_id` are required.
- The node restarts from its top: idempotent code before the interrupt, no `try/except` around it, stable order.
- `mastery.graphs.hitl.approval_node` = approve / edit / reject + validation loop + give-up; `run_until_pause` / `resume` / `drive` drive it.
- Parallel interrupts resume with `{id: answer}`; static breakpoints are for debugging.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 08.04 · Time travel: replay and fork** → [`08-04-time-travel.ipynb`](08-04-time-travel.ipynb).

[Module 08 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)